# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Select the features we identified in the previous assignment
features = ['word_count', 'char_count', 'search_volume', 'competition', 'cpc', 'content_type']

# Handle missing values: Fill missing numeric SEO metrics with 0, and word counts with median
df['search_volume'] = df['search_volume'].fillna(0)
df['competition'] = df['competition'].fillna(0)
df['cpc'] = df['cpc'].fillna(0)
df['word_count'] = df['word_count'].fillna(df['word_count'].median())
df['char_count'] = df['char_count'].fillna(df['char_count'].median())

# Handle categoricals: One-hot encode the 'content_type' column
X = pd.get_dummies(df[features], columns=['content_type'], drop_first=True)

print(f"Feature vector successfully built with {X.shape[0]} rows and {X.shape[1]} columns.")

# Show the first few rows of our clean feature vector
X.head()

Feature vector successfully built with 30000 rows and 7 columns.


,word_count,char_count,search_volume,competition,cpc,content_type_feedly article,content_type_keyword article
0,3221.0,20457.0,10.0,0.67,2.05,False,True
1,2481.0,15562.0,90.0,0.01,0.05,False,True
2,3515.0,23643.0,0.0,0.00,0.00,False,True
3,2877.0,19116.0,10.0,0.00,0.00,False,True
4,2803.0,17469.0,0.0,0.00,0.00,False,True


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

word_count & char_count: These represent the length of the content. Missing values were filled with the median to avoid skewing the data. Availability: These are fully available at prediction time (before the content is even published).

search_volume, competition, cpc: These are SEO keyword metrics. Missing values were filled with 0 (assuming no data means zero volume/competition). Availability: These are available at prediction time via external SEO tools.

content_type: This categorical variable tells us the format of the content. It was handled using one-hot encoding (converting categories into binary 1/0 columns). Availability: Available at prediction time since the format is decided before creation.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Future-Window Leakage: The FlyRank dictionary warns that _90d metrics (like impressions_90d) overlap with the snapshot's final months. If our label is based on the last month, using 90-day metrics means the model is peeking into the future.

Product Flags: Columns like health_score are assigned by internal systems after content is evaluated. Using this would be a direct label leak.

In [2]:
# List of known leaky or target columns we MUST avoid
leaky_columns = ['health_score', 'impressions_90d', 'clicks_90d', 'position_tier']

# Test if any of these snuck into our clean feature vector (X) from Section 1
leakage_found = [col for col in leaky_columns if col in X.columns]

if len(leakage_found) == 0:
    print("✅ Leakage test passed: No future-window or product flag columns found in the feature vector X.")
else:
    print(f"❌ WARNING: Leakage detected! Found these columns in X: {leakage_found}")

✅ Leakage test passed: No future-window or product flag columns found in the feature vector X.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded Fields & Justification:

content_id / URLs: Excluded because these are unique identifiers. The model would just memorize specific pages instead of learning underlying patterns.

client_id: Excluded to prevent client bias and respect data privacy boundaries.

impressions_90d / clicks_90d: Excluded due to target overlap (Data Leakage). We cannot use the outcome to predict the outcome.

health_score: Excluded because it is a downstream product flag generated after performance is already known.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.